# Establecer conexión con SQL

In [1]:
import mysql.connector
from mysql.connector import Error
import pandas as pd

In [3]:
try:
    connection = mysql.connector.connect(host='212.227.90.6',
                                         database='Equip_15',
                                         user='Equipo15',
                                         password = 'E1q2u3i4p5o15')
    if connection.is_connected():
        db_Info = connection.get_server_info()
        print("Connected to MySQL Server version ", db_Info)
        
    RRHH = pd.read_sql(f"SELECT * FROM RRHH_15092025", connection)

except Error as e:
    print("Error while connecting to MySQL", e)

Connected to MySQL Server version  8.0.43-0ubuntu0.24.04.1


C:\Users\xXSrBiscuitXx\AppData\Local\Temp\ipykernel_37564\687057739.py:7: DeprecationWarning: Call to deprecated function get_server_info. Reason: 
    The property counterpart 'server_info' should be used instead.

  db_Info = connection.get_server_info()
C:\Users\xXSrBiscuitXx\AppData\Local\Temp\ipykernel_37564\687057739.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  RRHH = pd.read_sql(f"SELECT * FROM RRHH_15092025", connection)


In [3]:
pd.set_option("display.max.columns", None)
#RRHH.describe() #variables numèriques
#RRHH.describe(include = 'object') #variables categòriques
RRHH.dtypes

ID                          int64
Reason_absence              int64
Month_absence               int64
Day_week                    int64
Seasons                     int64
Transportation_expense      int64
Distance_Residence_Work     int64
Service_time                int64
Age                         int64
Work_load_Average_day      object
Hit_target                  int64
Disciplinary_failure       object
Education                  object
Son                        object
Social_drinker             object
Social_smoker              object
Pet                        object
Weight                      int64
Height                      int64
Body_mass_index             int64
Absenteeism_hours           int64
dtype: object

# Cambiar los tipos de datos

In [ ]:
# Change from numeric to categorical
RRHH["Reason_absence"] = RRHH["Reason_absence"].astype("category")

# Change from object to numeric
for col in ["Son", "Pet"]:
    RRHH[col] = RRHH[col].astype("int")

# Change decimal separator and change type to float
RRHH["Work_load_Average_day"] = (
    RRHH["Work_load_Average_day"]
    .str.replace(",", ".", regex=False) 
    .astype(float)
)

#Normalizado
RRHH["Hit_target"] = (
    RRHH["Hit_target"].astype(float)
)

RRHH.dtypes

In [6]:
# Especificar meses
month_map = {
    1: "Enero", 2: "Febrero", 3: "Marzo", 4: "Abril",
    5: "Mayo", 6: "Junio", 7: "Julio", 8: "Agosto",
    9: "Septiembre", 10: "Octubre", 11: "Noviembre", 12: "Diciembre"
}

RRHH["Month_absence"] = RRHH["Month_absence"].replace(month_map)

# Especificar días de la semana
day_map = {
    2: "Lunes", 3: "Martes", 4: "Miercoles", 5: "Jueves", 6: "Viernes"
}
RRHH["Day_week"] = RRHH["Day_week"].replace(day_map)

# Especificar estaciones (cambiando invierno por verano y primavera por otoño, etc.)
spanish_season_map = {
    1: "Invierno", 2: "Otono", 3: "Verano", 4: "Primavera"
}
RRHH["Seasons"] = RRHH["Seasons"].replace(spanish_season_map)

In [6]:
RRHH.describe()
#RRHH.describe(include = 'object') 
#RRHH.describe(include = 'category') 

,ID,Transportation_expense,Distance_Residence_Work,Service_time,Age,Work_load_Average_day,Hit_target,Son,Pet,Weight,Height,Body_mass_index,Absenteeism_hours
count,740.000000,740.000000,740.000000,740.000000,740.000000,740.000000,740.000000,740.000000,740.000000,740.000000,740.000000,740.000000,740.000000
mean,18.017568,221.329730,29.631081,12.554054,36.450000,271.490235,94.587838,1.018919,0.745946,79.035135,172.114865,26.677027,6.924324
std,11.021247,66.952223,14.836788,4.384873,6.478772,39.058116,3.779313,1.098489,1.318258,12.883211,6.034995,4.285452,13.330998
min,1.000000,118.000000,5.000000,1.000000,27.000000,205.917000,81.000000,0.000000,0.000000,56.000000,163.000000,19.000000,0.000000
25%,9.000000,179.000000,16.000000,9.000000,31.000000,244.387000,93.000000,0.000000,0.000000,69.000000,169.000000,24.000000,2.000000
50%,18.000000,225.000000,26.000000,13.000000,37.000000,264.249000,95.000000,1.000000,0.000000,83.000000,170.000000,25.000000,3.000000
75%,28.000000,260.000000,50.000000,16.000000,40.000000,294.217000,97.000000,2.000000,1.000000,89.000000,172.000000,31.000000,8.000000
max,36.000000,388.000000,52.000000,29.000000,58.000000,378.884000,100.000000,4.000000,8.000000,108.000000,196.000000,38.000000,120.000000


# Eliminar duplicados exactos

In [7]:
RRHH = RRHH.drop_duplicates()
RRHH.shape[0]

706

# Exportar a csv

In [8]:
RRHH.to_csv("RRHH.csv", index=False, sep=",")

# Cerrar conexión

In [9]:
connection.close()

# Transformación datos


In [8]:
#Normalizar onjetivos de entrega
RRHH.loc[:, 'Hit_target'] = RRHH['Hit_target'] / 100

#Borrar info redundante
RRHH.drop('Body_mass_index', axis=1, inplace=True)

#Dejo la nueva columna para más tarde pero serán los grupos sociodemograficos que definamos

#Nueva columna

# Clustering

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Agrupar por ids
socio = RRHH.groupby("ID").first()[["Age","Distance_Residence_Work","Transportation_expense" ,"Son", "Education", "Weight", "Height", "Pet", "Social_drinker", "Social_smoker"]]

# Adaptar las variables categoricas
categorical_cols = ["Education", "Social_drinker", "Social_smoker"]
for col in categorical_cols:
    socio[col] = socio[col].astype(str)
    socio[col] = socio[col].fillna("Desconocido")
df_socio = pd.get_dummies(socio, columns=categorical_cols, drop_first=True)

# Escalado
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_socio)

# Numero de clusters y resultados
kmeans = KMeans(n_clusters=4, random_state=42)
clusters = kmeans.fit_predict(X_scaled)
df_socio["cluster"] = clusters
cluster_summary = df_socio.groupby("cluster").mean()
print(cluster_summary)

# Ids de cada grupo
cluster_ids = {}
for c in sorted(df_socio["cluster"].unique()):
    ids = df_socio[df_socio["cluster"] == c].index.tolist()
    cluster_ids[c] = ids
    print(f"\nCluster {c} ({len(ids)} empleados):")
    print(ids)

               Age  Distance_Residence_Work  Transportation_expense       Son  \
cluster                                                                         
0        37.500000                13.500000              190.000000  2.000000   
1        41.454545                24.454545              262.818182  1.818182   
2        41.000000                34.230769              236.538462  0.923077   
3        31.600000                23.800000              219.600000  0.500000   

            Weight      Height       Pet  Education_2  Education_3  \
cluster                                                              
0        94.500000  189.000000  1.000000     0.000000          0.0   
1        71.545455  172.454545  1.000000     0.090909          0.0   
2        87.461538  170.846154  1.538462     0.000000          0.0   
3        72.900000  173.400000  1.300000     0.300000          0.3   

         Education_4  Social_drinker_1  Social_smoker_1  
cluster                           

# KPYS

1. Tasa mensual de absentismo per cápita:
Suma de horas de absentismo en el mes 
(DIVIDIDO ENTRE) 
Numero de empleados activos en el mes

In [11]:
#Calculo primer kpy

2. Mes pico de carga por ausencias:
El mes con la maxima suma de horas de absentismo

In [12]:
# Agrupar por mes y sumar las horas de absentismo
absentismo_por_mes = RRHH.groupby('Month_absence')['Absenteeism_hours'].sum()

# Encontrar el mes con el máximo valor
mes_pico = absentismo_por_mes.idxmax()
max_horas = absentismo_por_mes.max()

print(f"El mes pico de carga por ausencias es {mes_pico} con {max_horas} horas de absentismo.")

El mes pico de carga por ausencias es Marzo con 755 horas de absentismo.


3. Motivo modal de absentismo laboral:
Maxima frequencia de motivo de ausencia


In [13]:
reason_absence_dict = {
    0: "Sin especificar / Otros",
    1: "Enfermedades infecciosas y parasitarias",
    2: "Neoplasias (tumores)",
    3: "Enfermedades de la sangre y órganos hematopoyéticos",
    4: "Enfermedades endocrinas, nutricionales y metabólicas",
    5: "Trastornos mentales y del comportamiento",
    6: "Enfermedades del sistema nervioso",
    7: "Enfermedades del ojo y sus anexos",
    8: "Enfermedades del oído y la apófisis mastoides",
    9: "Enfermedades del sistema circulatorio",
    10: "Enfermedades del sistema respiratorio",
    11: "Enfermedades del sistema digestivo",
    12: "Enfermedades de la piel y tejido subcutáneo",
    13: "Enfermedades del sistema musculoesquelético y tejido conectivo",
    14: "Enfermedades del sistema genitourinario",
    15: "Embarazo, parto y puerperio",
    16: "Condiciones originadas en el período perinatal",
    17: "Malformaciones congénitas y anomalías cromosómicas",
    18: "Síntomas y signos no clasificados",
    19: "Lesiones, envenenamientos y consecuencias de otras causas externas",
    20: "Causas externas de morbilidad y mortalidad",
    21: "Factores que influyen en el estado de salud",
    22: "Paciente en seguimiento",
    23: "Consulta médica",
    24: "Donación de sangre",
    25: "Examen de laboratorio",
    26: "Ausencia injustificada",
    27: "Fisioterapia",
    28: "Consulta dental"
}
# Calcular el motivo modal de absentismo
motivo_modal = RRHH['Reason_absence'].mode()[0]
frecuencia_modal = RRHH['Reason_absence'].value_counts().max()

# Obtener la descripción del motivo modal del diccionario
descripcion_modal = reason_absence_dict.get(motivo_modal, "Motivo desconocido")

# Imprimir el resultado
print(f"El motivo modal de absentismo es: {descripcion_modal} (Código: {motivo_modal})")
print(f"Frecuencia: {frecuencia_modal} ausencias")

El motivo modal de absentismo es: Consulta médica (Código: 23)
Frecuencia: 142 ausencias


4. Indice de eficencia relativa por carga laboral:
Suma de hit target 
(DIVIDIDO ENTRE) 
Suma de work load average/day


In [ ]:
suma_hit = RRHH['Hit_target'].sum()        # sigue siendo 0-1
suma_workload = RRHH['Work_load_Average_day'].sum()

indice_eficiencia = (suma_hit / suma_workload)
print("Índice eficiencia:", indice_eficiencia)

#Por cada paquete de media diaria, se consigue el 0.003474885664829815 de hit target

Índice eficiencia: 0.003474885664829815
